In [1]:
# RUN
!pip install unsloth --q
# Also get the latest nightly Unsloth!
# !pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git --q
# !pip install vllm
%load_ext autoreload
%autoreload 2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.6/191.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.1/253.1 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from datasets import load_dataset,Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import GRPOConfig, GRPOTrainer
from transformers import (
AutoModelForCausalLM, 
AutoTokenizer, 
Qwen2_5_VLForConditionalGeneration, 
AutoProcessor,
BitsAndBytesConfig
)
import torch
compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)
tokenizer = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct",
                                         trust_remote_code=True)
# use cuda device
model = Qwen2_5_VLForConditionalGeneration.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", 
                                             device_map="auto", 
                                             trust_remote_code=True,
                                            torch_dtype=compute_dtype,
                                            quantization_config=bnb_config).eval()


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

Single inference

In [3]:
# RUN
from datasets import load_dataset,Dataset
from PIL import Image
import base64
from io import BytesIO
import pandas as pd
from tqdm import tqdm

ds = load_dataset("BUAADreamer/llava-med-zh-instruct-60k",split = "train[0:2000]", trust_remote_code=True)
print(ds[0])# show content

def encode_image(image):
    """将 PIL 图片转换为 base64 编码字符串"""
    buffered = BytesIO()
    image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

def get_prompt_rft(example):
    '''
    input: dict example, including PIL image object
    output: multiple samples, within a dict format, like: [
    {'image': 'data:image;base64,/9j/...'},
    {'text': '这是什么'},
]
    '''
    dialogue_num = len(example['messages'])
    i = 0
    results=[]
    while i<dialogue_num:
        assert example['messages'][i]['role']=='user' and example['messages'][i+1]['role']=='assistant'
        question_sample = example['messages'][i]['content']
        answer_sample = example['messages'][i+1]['content'] # ?
        img_pil = example['images'][0].resize((128,128))  # reduce vRAM burden
        # image_b64 = encode_image(img_pil)
        out_results = []
        SYSTEM_PROMPT = r'''
        Below is an instruction that describes a task, paired with an input that provides further context.
        Write a response that appropriately completes the request.
        Before answering, think carefully about the question and create a step-by-step chain of 
        thoughts to ensure a logical and accurate response.
        
        ### Instruction:
        You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
        Please answer the following medical question based on the input image. Output the thinking process in <think> </think> and final answer in <answer> </answer> tags.The output answer format should be as follows:
<think> ... </think> 除了特殊符号，请用中文回答
        '''.strip()   # for a different language, please change the last few words.
        results.append({
                'prompt': [
                    {'role': 'system', 'content': [{"type": "text", "text": SYSTEM_PROMPT}]},
                    {'role': 'user', 'content': [
                        {"type": "image", },  
                        {"type": "text", "text": question_sample},    
                    ]}
                ],
                'image':img_pil,
                'solution':answer_sample,
            })
        i+=2
    return results

def get_prompt_sft(example):
    '''
    input: dict example, including PIL image object
    output: multiple samples, within a dict format, like: [
    {'image': 'data:image;base64,/9j/...'},
    {'text': '这是什么'},
]
    '''
    dialogue_num = len(example['messages'])
    i = 0
    results=[]
    while i<dialogue_num:
        assert example['messages'][i]['role']=='user' and example['messages'][i+1]['role']=='assistant'
        question_sample = example['messages'][i]['content']
        answer_sample = example['messages'][i+1]['content'] # ?
        image_pil = example['images'][0]
        # image_b64 = encode_image(image_pil)
        SYSTEM_PROMPT = r'''
        Below is an instruction that describes a task, paired with an input that provides further context.
        Write a response that appropriately completes the request.
        Before answering, think carefully about the question and create a step-by-step chain of 
        thoughts to ensure a logical and accurate response.
        
        ### Instruction:
        You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
        Please answer the following medical question based on the input image. 
        '''.strip()
        TEXT_FORMAT = f'''
        ### Answer:
        {answer_sample}
        '''.strip()
        # image_prompt = f'data:image;base64,{image_b64}'
        results.append({'prompt': [
                {'role': 'system', 'content': [{"type": "text", "text": SYSTEM_PROMPT}]},
                {'role': 'user', 'content': [
                    {"type": "image", "image": image_pil}, 
                    {"type": "text", "text": question_sample},    
                ]},
                {'role': 'assistant', 'content': [{"type": "text", "text": TEXT_PROMPT}]},
            ],
            'solution':[answer_sample]})
        i+=2
    return results

README.md:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

train-00000-of-00014.parquet:   0%|          | 0.00/444M [00:00<?, ?B/s]

train-00001-of-00014.parquet:   0%|          | 0.00/450M [00:00<?, ?B/s]

train-00002-of-00014.parquet:   0%|          | 0.00/458M [00:00<?, ?B/s]

train-00003-of-00014.parquet:   0%|          | 0.00/444M [00:00<?, ?B/s]

...

train-00011-of-00014.parquet:   0%|          | 0.00/492M [00:00<?, ?B/s]

train-00012-of-00014.parquet:   0%|          | 0.00/474M [00:00<?, ?B/s]

train-00013-of-00014.parquet:   0%|          | 0.00/484M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/56649 [00:00<?, ? examples/s]

{'messages': [{'role': 'user', 'content': '图中的组织类型是什么？'}, {'role': 'assistant', 'content': '图中的组织切片显示了一个鼻内肿块的病理学样本，这是从鼻腔内部取样的。'}, {'role': 'user', 'content': '放大倍率是多少？'}, {'role': 'assistant', 'content': '图像的放大倍数为200倍。'}, {'role': 'user', 'content': '根据病理学特征，诊断是什么？'}, {'role': 'assistant', 'content': '诊断为B细胞淋巴瘤。B细胞淋巴瘤起源于B淋巴细胞，这是一种白血细胞。影像中显示，大量圆形大细胞弥漫性地遮盖了呼吸道上皮下的基底膜。结合其他临床和实验室资料，这一诊断得以确立。'}], 'images': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=720x494 at 0x7D67981547C0>]}


Try Inference

In [4]:
# RUN
# FastVisionModel.for_inference(model) # Enable for inference!

image = ds[0]['images'][0]
instruction = "You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. \
Please answer the following medical question based on the input image. 请用中文回答"
# for a different language, please change the last few words.
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

这张显微镜下的图像显示的是组织切片，染色后观察到细胞核和细胞质。根据图像的特征，可以看到大量的细胞核密集分布，细胞形态较为一致，且没有明显的异型性或病理性改变。这种类型的图像通常用于病理诊断，以帮助识别疾病状态。

然而，仅凭这张图片无法做出确切的诊断。病理诊断需要结合临床症状、病史和其他实验室检查结果进行综合分析。如果这是来自一个疑似肿瘤的组织样本，可能需要进一步的免疫组化染色或其他分子生物学检测来确定具体的病理类型。

如果你有更多关于这个病例的信息（如患者的年龄、性别、症状等），或者需要进一步的帮助，请提供详细信息，我会尽力为你提供更准确的建议。<|im_end|>


In [5]:
# RUN
# All reward functions for GRPO
import re
!pip install levenshtein
from Levenshtein import ratio as levenshtein_ratio
def format_reward_func(completions, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    # print(completions) #debug
    pattern = r"^<think>.*?</think>.*?<answer>.*?</answer>$"
    matches = [re.match(pattern, content[0]['content'], re.DOTALL) for content in completions]
    return [1.0 if match else 0.0 for match in matches]
def levenshtein_reward_func(completions, solution, **kwargs):
    """Reward function that checks if the completion get solutions correctly."""
    res = []
    for completion, sol in zip(completions, solution):
        completion = completion[0]['content']
        if '</think>' in completion:
            t = completion.split('</think>')[-1]    # calculate result distance
            res.append(levenshtein_ratio(t, sol))
        else:
            res.append(0.0)
    return res

def dataset_gen():
    for items in ds:
        multiple_out = get_prompt_rft(items)
        for single_out in multiple_out:
            yield single_out
my_gen = dataset_gen()

dataset_train = Dataset.from_generator(dataset_gen)
print(dataset_train[-1])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.4 MB/s eta 0:00:00


Generating train split: 0 examples [00:00, ? examples/s]

{'prompt': [{'content': [{'text': 'Below is an instruction that describes a task, paired with an input that provides further context.\n        Write a response that appropriately completes the request.\n        Before answering, think carefully about the question and create a step-by-step chain of \n        thoughts to ensure a logical and accurate response.\n        \n        ### Instruction:\n        You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.\n        Please answer the following medical question based on the input image. Output the thinking process in <think> </think> and final answer in <answer> </answer> tags.The output answer format should be as follows:\n<think> ... </think> 除了特殊符号，请用中文回答', 'type': 'text'}], 'role': 'system'}, {'content': [{'text': None, 'type': 'image'}, {'text': '胸腔穿刺的目的是什么？', 'type': 'text'}], 'role': 'user'}], 'image': <PIL.PngImagePlugin.PngImageFile image mode=L size=128x128 at 0x7D67842DC490

In [6]:
output_dir="./outputs/Qwevl-Instruct-GRPO"
run_name="Qwen-vl-GRPO-medical"
# from unsloth import is_bfloat16_supported
from trl import GRPOConfig
!git clone https://github.com/auto-Dog/vlm_rft_trainer.git
!cd /kaggle/working/vlm_rft_trainer && git pull
!cp /kaggle/working/vlm_rft_trainer/grpo_trainer.py grpo_trainer.py

from grpo_trainer import Qwen2VLGRPOTrainer # third-party trainer from open-R1

Cloning into 'vlm_rft_trainer'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 85 (delta 27), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 35.26 KiB | 7.05 MiB/s, done.
Resolving deltas: 100% (27/27), done.
Already up to date.


In [7]:
model.train()
peft_config = LoraConfig(
    r=32, #Rank
    lora_alpha=16,
    target_modules=[
        "q_proj", 
        "k_proj", 
        "v_proj", 
        "o_proj", 
        # "gate_proj", 
        # "up_proj", 
        # "down_proj"
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
)

training_args = GRPOConfig(
    # use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    bf16 = False,
    fp16 = True,
    per_device_train_batch_size = 1,# keep same with num_generations
    gradient_accumulation_steps = 2, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = 2048,
    max_completion_length = 2048,
    num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 5,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)
trainer = Qwen2VLGRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func, # all reward functions
        levenshtein_reward_func],
    args=training_args,
    train_dataset=dataset_train,
    peft_config = peft_config,
)

trainer.train()

trainer.save_model(output_dir)

No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
1,0.000000
2,-0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,-0.000000
9,0.000000
10,0.000000


Inference after training

In [8]:
# Preparation for inference
model.eval()
message = dataset_train[0]['prompt']
image = dataset_train[0]['image']
input_text = tokenizer.apply_chat_template(message, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

<think>这张图片显示的是一个显微镜下的组织切片。从图像中可以看到细胞排列紧密，细胞核较大且染色较深，这通常与某些类型的肿瘤或炎症反应相关。然而，仅凭这张图片无法准确诊断出具体的疾病类型。需要结合临床症状、病史和其他实验室检查结果来综合判断。

在病理学中，这种细胞排列和染色模式可能提示上皮性病变，如腺癌或鳞状细胞癌。但是，为了做出准确的诊断，还需要更多的信息，包括患者的年龄、性别、临床表现以及是否有家族遗传病史等。

因此，我的最终答案是：这张图片显示的是一个显微镜下的组织切片，但具体组织类型尚不能确定。</think>
<answer>这张图片显示的是一个显微镜下的组织切片，但具体组织类型尚不能确定。</answer><|im_end|>


Utils: to empty cache in GPU memory you may:

In [9]:
del Qwen2VLGRPOTrainer
# del model,tokenizer
torch.cuda.empty_cache()